# 🔌 IE Iberdrola Datathon 2026 — Master Dataset Pipeline
## `master_dataset_interurban.parquet / .geojson`

**One row = one candidate charging station point**, generated every 50 km along each interurban road in Spain.

### Input datasets (real column names)
| Source notebook | DataFrame | Key columns |
|---|---|---|
| `IMD_cleaned` | `imd_interurban_cleaned.parquet` | `road_code`, `imd_total`, `imd_light`, `centroid_lat`, `centroid_lon`, geometry |
| `roads_cleaned` | `df_roads` | `carretera`, `clase`, `provincia`, `is_highway`, geometry |
| `charging_points_cleaned` | `charging_points_final_features.csv` | `site_key`, `lat`, `lon`, `number_of_connections`, `total_power_kw` |
| `grid_demand_cleaned` | `df_grid_clean` | `latitude`, `longitude`, `capacity_mw`, `capacity_kw` |
| `EV_prediction_cleaned` | `df_ev` | `categoría_vehículo_eléctrico`, `year`, `year_month` (no province column) |

### Pipeline
```
Step 0  → Installation & Imports
Step 1  → Global Configuration & Parameters
Step 2  → Load Clean Datasets
Step 3  → Candidate Point Generation (50 km spacing on roads_cleaned)
Step 4  → Spatial Join: provinces + distributor assignment
Step 5  → IMD Join: traffic to nearest candidate point
Step 6  → EV Adoption: DGT proxy by province → traffic-weighted disaggregation
Step 7  → Existing Infrastructure: distance to nearest charging point
Step 8  → Grid Capacity: nearest substation (i-DE)
Step 9  → Feature Engineering: demand, grid_status, priority_score
Step 10 → Validation & Export of master dataset
```

---
## Step 0 — Installation & Imports

> 📌 **What this step does:** **Step 0** installs all required Python libraries and sets global display options. It also auto-detects the project root directory by walking up from the current working directory until it finds the `Data/Raw` and `Data/Processed` folders, making paths portable across machines and Colab.

In [1]:
# Uncomment when running on Google Colab:
# !pip install geopandas pyproj shapely fiona scipy scikit-learn pyarrow --quiet

import os, math, warnings
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import cKDTree
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print(f'geopandas : {gpd.__version__}')
print(f'pandas    : {pd.__version__}')
print('✅ Imports OK')

geopandas : 1.1.3
pandas    : 2.2.3
✅ Imports OK


In [2]:
from pathlib import Path

ROOT = Path.cwd().resolve()

while not (
    (ROOT / "Data" / "Processed").exists() and
    (ROOT / "Data" / "Raw").exists()
) and ROOT.parent != ROOT:
    ROOT = ROOT.parent

if not (ROOT / "Data").exists():
    raise ValueError("Project root not found")

DATA = ROOT / "Data"
RAW = DATA / "Raw"
PROCESSED = DATA / "Processed"

print("ROOT:", ROOT)

ROOT: C:\Users\alber\OneDrive\ALBERTO\Iberdrola_Datathon


---
## Step 1 — Global Configuration & Parameters

**All model parameters are centralised here.** Changing a value here propagates through the entire pipeline.

> 📌 **What this step does:** **Step 1** defines every modelling parameter in one place — spacing between candidate points, charger power, EV charging probability, simultaneity factor, grid thresholds, and file paths. All assumptions are documented inline with their ID (A1–A10) and justification. Changing any value here automatically propagates to every downstream step.

In [3]:
# ─────────────────────────────────────────────────────────────
# CONFIGURACIÓN GLOBAL — Assumptions documentadas (ver Informe)
# ─────────────────────────────────────────────────────────────

# Coordinate reference systems
CRS_GEO  = 'EPSG:4326'   # WGS84 — coordinates for outputs
CRS_PROJ = 'EPSG:25830'  # UTM zone 30N — distances in metres (mainland Spain)

# A1: Spacing between candidate points
# Justification: min range 300 km × 60% usable / 2 = 90 km max without charge.
# 50 km is conservative — ensures no driver is ever stranded.
SPACING_M = 50_000

# A4: Minimum power to classify as real interurban coverage (AFIR 2025)
MIN_FAST_KW = 50

# A6: Probability that an EV passing a given point will charge there
# Source: ICCT 2023 — 1.2 cargas/100km en viajes largos → 50km/100km × 1.2 = 0.08
CHARGE_PROB = 0.08

# A7: Peak-hour simultaneity factor
# Spanish peak hour = 10-12% of daily traffic; DC session = ~20 min
# → 0.11 × (20/60) × 2 overlapping sessions = ~0.25
SIMULTANEITY = 0.25

# Fixed datathon rule
KW_PER_CHARGER = 150

# A8: Target charger utilisation rate
# <50% → under-invested; >80% → unacceptable queuing. 65% = balance point.
TARGET_UTIL = 0.65

# Charger count limits per station
MIN_CHARGERS = 2
MAX_CHARGERS = 12

# A9: grid_status thresholds based on nearest substation
# Ref: estación de 6 cargadores × 150 kW = 900 kW × 1.5 safety margin = 1.35 MW → 1.5 MW mínimo
GRID_SUFFICIENT_MW = 5.0
GRID_MODERATE_MW   = 1.5
LOAD_RATIO_MODERATE   = 0.30
LOAD_RATIO_CONGESTED  = 0.80

# A5: Maximum reliable distance for substation assignment
SUBSTATION_MAX_KM = 15.0

# A10: Priority score weights
W_DEMAND = 0.40
W_GAP    = 0.35
W_GRID   = 0.25

# Target year
TARGET_YEAR = 2027

# ─── FILE PATHS — CORRECT AND PORTABLE ───

# Processed
PATH_IMD = PROCESSED / "IMD_cleaned" / "imd_interurban_cleaned.parquet"
PATH_GRID = PROCESSED / "grid_cleaned" / "grid_clean.parquet"
PATH_CHARGERS = PROCESSED / "charging_points_cleaned" / "charging_points_final_features.csv"
PATH_EV_DIR = PROCESSED / "ev_cleaned" / "ev_clean.parquet"
PATH_ROADS = PROCESSED / "roads_cleaned" / "roads_clean.parquet" 


# Output directory
OUT_DIR = PROCESSED / "master_dataset"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Debug: verify paths exist
print('✅ Configuración cargada')
print("PATH_IMD:", PATH_IMD, "| exists:", PATH_IMD.exists())
print("PATH_ROADS:", PATH_ROADS, "| exists:", PATH_ROADS.exists())
print("PATH_CHARGERS:", PATH_CHARGERS, "| exists:", PATH_CHARGERS.exists())
print("PATH_GRID:", PATH_GRID, "| exists:", PATH_GRID.exists())
print("PATH_EV_DIR:", PATH_EV_DIR, "| exists:", PATH_EV_DIR.exists())

print(f'   Espaciado candidatos : {SPACING_M/1000:.0f} km')
print(f'   kW por cargador      : {KW_PER_CHARGER} kW (fijo datathon)')
print(f'   Año objetivo         : {TARGET_YEAR}')
print(f'   Umbrales grid        : Sufficient ≥{GRID_SUFFICIENT_MW} MW | Moderate ≥{GRID_MODERATE_MW} MW')

✅ Configuración cargada
PATH_IMD: C:\Users\alber\OneDrive\ALBERTO\Iberdrola_Datathon\Data\Processed\IMD_cleaned\imd_interurban_cleaned.parquet | exists: True
PATH_ROADS: C:\Users\alber\OneDrive\ALBERTO\Iberdrola_Datathon\Data\Processed\roads_cleaned\roads_clean.parquet | exists: True
PATH_CHARGERS: C:\Users\alber\OneDrive\ALBERTO\Iberdrola_Datathon\Data\Processed\charging_points_cleaned\charging_points_final_features.csv | exists: True
PATH_GRID: C:\Users\alber\OneDrive\ALBERTO\Iberdrola_Datathon\Data\Processed\grid_cleaned\grid_clean.parquet | exists: True
PATH_EV_DIR: C:\Users\alber\OneDrive\ALBERTO\Iberdrola_Datathon\Data\Processed\ev_cleaned\ev_clean.parquet | exists: True
   Espaciado candidatos : 50 km
   kW por cargador      : 150 kW (fijo datathon)
   Año objetivo         : 2027
   Umbrales grid        : Sufficient ≥5.0 MW | Moderate ≥1.5 MW


---
## Step 2 — Load Clean Datasets

We load the exact outputs from the cleaning notebooks, preserving their real column names.

> 📌 **What this step does:** **Step 2** loads the five cleaned datasets produced by the preprocessing notebooks: IMD traffic data (`gdf_imd`, 506 road segments), the road network (`gdf_roads`, 6,411 segments), existing charging points (`gdf_chargers`, 12,072 sites), grid substation capacity (`gdf_grid`, 224 substations), and EV registration data (`df_ev`, 3.36M records). Each dataset is validated for CRS, shape, and key column availability before proceeding.

In [4]:
print("PATH_IMD:", PATH_IMD)
print("Parquet exists:", PATH_IMD.exists())
print("GeoJSON exists:", PATH_IMD.with_suffix('.geojson').exists())

PATH_IMD: C:\Users\alber\OneDrive\ALBERTO\Iberdrola_Datathon\Data\Processed\IMD_cleaned\imd_interurban_cleaned.parquet
Parquet exists: True
GeoJSON exists: True


In [5]:
# ─── 2.1 IMD — imd_interurban_cleaned ────────────────────────
# Output from IMD_cleaned.ipynb
# Real columns: road_code, pk_start, pk_end, length_m, length_km,
#                owner, imd_total, imd_light, imd_heavy, pct_heavy,
#                demand_index, centroid_lat, centroid_lon + geometry
# Note: NO road_type or segment_id — these are derived from roads_cleaned

try:
    gdf_imd = gpd.read_parquet(PATH_IMD)
    print('IMD loaded from parquet')
except Exception:
    path_geojson = PATH_IMD.with_suffix('.geojson')
    gdf_imd = gpd.read_file(path_geojson)
    print('IMD loaded from GeoJSON')

# Ensure correct CRS
if gdf_imd.crs and gdf_imd.crs.to_epsg() != 4326:
    gdf_imd = gdf_imd.to_crs(CRS_GEO)

print(f'Shape IMD  : {gdf_imd.shape}')
print(f'CRS        : {gdf_imd.crs}')
print(f'Columnas   : {gdf_imd.columns.tolist()}')
gdf_imd[['road_code','imd_total','imd_light','length_km','centroid_lat','centroid_lon']].head(3)

IMD loaded from GeoJSON
Shape IMD  : (506, 16)
CRS        : EPSG:4326
Columnas   : ['road_code', 'pk_start', 'pk_end', 'length_m', 'length_km', 'owner', 'imd_total', 'imd_light', 'imd_heavy', 'pct_heavy', 'demand_index', 'centroid_lat', 'centroid_lon', 'imd_total_outlier_flag', 'spatial_dup_flag', 'geometry']


,road_code,imd_total,imd_light,length_km,centroid_lat,centroid_lon
0,A-31,14075,11964.0000,0.7530,38.7717,-0.9607
1,A-31,14075,11964.0000,0.9910,38.7641,-0.9579
2,A-31,14075,11964.0000,2.9980,38.7472,-0.9502


In [6]:
# ─── 2.2 Roads — roads_cleaned ───────────────────────────────
# Output from roads_cleaned.ipynb
# Real columns: provincia, carretera, clase, is_highway, geometry
# clase values: 'AUTOVÍA', 'CARRETERA CONVENCIONAL' (no AUTOPISTA in data)
# CRS: EPSG:25830

gdf_roads = gpd.read_file(PATH_ROADS)
gdf_roads = gdf_roads[['provincia','carretera','clase','geometry']].copy()

# Normalise text (replicating roads_cleaned.ipynb)
gdf_roads['clase']     = gdf_roads['clase'].astype(str).str.strip().str.upper()
gdf_roads['carretera'] = gdf_roads['carretera'].astype(str).str.strip().str.upper()
gdf_roads['provincia'] = gdf_roads['provincia'].astype(str).str.strip().str.upper()

# Filter to relevant road types
VALID_CLASES = ['AUTOPISTA', 'AUTOVÍA', 'CARRETERA CONVENCIONAL']
gdf_roads = gdf_roads[gdf_roads['clase'].isin(VALID_CLASES)].copy()

# is_highway feature (replicating roads_cleaned.ipynb)
gdf_roads['is_highway'] = gdf_roads['carretera'].str.startswith(('A-', 'AP-'))

# Drop duplicates and null geometries
gdf_roads = gdf_roads[gdf_roads['geometry'].notna()].drop_duplicates()

print(f'Shape Roads: {gdf_roads.shape}')
print(f'CRS        : {gdf_roads.crs}')
print(f'Clases     :\n{gdf_roads["clase"].value_counts()}')
gdf_roads.head(3)

Shape Roads: (6411, 5)
CRS        : EPSG:4326
Clases     :
clase
CARRETERA CONVENCIONAL    4344
AUTOVÍA                   2067
Name: count, dtype: int64


,provincia,carretera,clase,geometry,is_highway
0,MURCIA,A-30,AUTOVÍA,"LINESTRING Z (-1.26827 38.13374 132.6, -1.2675...",True
1,TERUEL,N-211A,CARRETERA CONVENCIONAL,"LINESTRING Z (-0.41835 40.88235 693, -0.41802 ...",False
2,LUGO,A-54,AUTOVÍA,"LINESTRING Z (-7.89326 42.87754 521.6462, -7.8...",True


In [7]:
# ─── 2.3 Charging Points — charging_points_final_features ────
# Output from charging_points_cleaned.ipynb
# Real columns: site_key, lat, lon, number_of_connections, total_power_kw
# 12,072 points across Spain (WGS84 coordinates, directly usable)

df_chargers = pd.read_csv(PATH_CHARGERS)

# Clean coordinates and build GeoDataFrame
df_chargers = df_chargers.dropna(subset=['lat', 'lon'])
df_chargers = df_chargers[
    df_chargers['lat'].between(27, 45) &
    df_chargers['lon'].between(-19, 5)
].copy()

gdf_chargers = gpd.GeoDataFrame(
    df_chargers,
    geometry=gpd.points_from_xy(df_chargers['lon'], df_chargers['lat']),
    crs=CRS_GEO
)

# A4: Flag fast chargers (≥50 kW) — real interurban coverage
gdf_chargers['is_fast_charger'] = (
    gdf_chargers['total_power_kw'].fillna(0) >= MIN_FAST_KW
)

print(f'Shape Chargers total : {gdf_chargers.shape}')
print(f'Con potencia conocida: {gdf_chargers["total_power_kw"].notna().sum()}')
print(f'Cargadores rápidos   : {gdf_chargers["is_fast_charger"].sum()}')
gdf_chargers[['site_key','lat','lon','number_of_connections','total_power_kw','is_fast_charger']].head(3)

Shape Chargers total : (12072, 7)
Con potencia conocida: 12072
Cargadores rápidos   : 12072


,site_key,lat,lon,number_of_connections,total_power_kw,is_fast_charger
0,hash::592f9cea72b4e0f6,42.7190,-2.9852,16,29440.0000,True
1,hash::905a43e5208100ec,41.5120,-5.7485,8,44000.0000,True
2,hash::7fa723e5fb1aeaf0,41.5066,-5.7286,8,22000.0000,True


In [9]:
# ─── 2.4 Grid — df_grid_clean ────────────────────────────────
# Output from grid_demand_cleaned.ipynb
# Real columns: latitude, longitude, capacity_mw, capacity_kw
# 224 i-DE substations (Iberdrola). UTM→WGS84 already converted.
# NOTE: Covers i-DE network only. Endesa and Viesgo must be added
#       following the same process in grid_demand_cleaned notebook.


try:
    df_grid = pd.read_parquet(PATH_GRID)
    print('Grid loaded from parquet')
except FileNotFoundError:
    # Option B: Rebuild inline from raw data (replicates grid_demand_cleaned.ipynb)
    print('parquet not found — rebuilding from raw data...')
    from pyproj import Transformer
    import glob

    grid_files = glob.glob('Data/Raw/Grid_Stations/*Demanda*.csv')
    dfs_g = []
    for f in grid_files:
        df_temp = pd.read_csv(f, sep=';', encoding='utf-8')
        df_temp = df_temp.rename(columns={
            'Coordenada UTM X': 'utm_x',
            'Coordenada UTM Y': 'utm_y',
            'Capacidad firme disponible (MW)': 'capacity_mw'
        })
        df_temp = df_temp.drop(columns=['Posiciones ocupadas','Posiciones libres','Comentarios'], errors='ignore')
        for col in ['utm_x','utm_y','capacity_mw']:
            if col in df_temp.columns:
                df_temp[col] = pd.to_numeric(
                    df_temp[col].astype(str).str.strip().str.replace('.',"",regex=False).str.replace(',','.',regex=False),
                    errors='coerce'
                )
        dfs_g.append(df_temp)

    df_grid = pd.concat(dfs_g, ignore_index=True)
    df_grid = df_grid.dropna(subset=['utm_x','utm_y','capacity_mw'])
    df_grid = df_grid[df_grid['capacity_mw'] > 0]

    t = Transformer.from_crs('EPSG:25830','EPSG:4326', always_xy=True)
    lon_g, lat_g = t.transform(df_grid['utm_x'].values, df_grid['utm_y'].values)
    df_grid['longitude'] = lon_g
    df_grid['latitude']  = lat_g
    df_grid = df_grid[df_grid['latitude'].between(35,45) & df_grid['longitude'].between(-10,5)]
    df_grid['capacity_kw'] = df_grid['capacity_mw'] * 1000
    df_grid = df_grid[['latitude','longitude','capacity_mw','capacity_kw']].drop_duplicates()

gdf_grid = gpd.GeoDataFrame(
    df_grid,
    geometry=gpd.points_from_xy(df_grid['longitude'], df_grid['latitude']),
    crs=CRS_GEO
)

print(f'Shape Grid : {gdf_grid.shape}')
print(f'capacity_mw: min={gdf_grid["capacity_mw"].min():.2f} | mean={gdf_grid["capacity_mw"].mean():.2f} | max={gdf_grid["capacity_mw"].max():.2f}')
gdf_grid.head(3)

Grid loaded from parquet
Shape Grid : (224, 5)
capacity_mw: min=0.15 | mean=13.63 | max=112.34


,latitude,longitude,capacity_mw,capacity_kw,geometry
153,38.7069,-0.4812,16.4200,16420.0000,POINT (-0.48122 38.70686)
155,38.7069,-0.4812,0.8400,840.0000,POINT (-0.48122 38.70686)
158,38.7193,-0.9237,46.0400,46040.0000,POINT (-0.92373 38.71926)


In [10]:
# 2.5 We create data frame for EV

df_ev = pd.read_parquet(PATH_EV_DIR)
df_ev.head()

,fec_matricula,marca_itv,modelo_itv,cod_tipo,cod_propulsion_itv,clave_tramite,categoría_vehículo_eléctrico,fecha_dataset,dataset_year_month,source_file,fecha_matricula,year_matricula,month_matricula,year_month_matricula,year
569,02012015,LEXUS,LEXUS GS300H,40,0,1,HEV,2015-01-01,2015-01,2015_01.parquet,2015-01-02,2015.0000,1.0000,2015-01,2015.0000
670,02012015,TOYOTA,AURIS,40,0,1,HEV,2015-01-01,2015-01,2015_01.parquet,2015-01-02,2015.0000,1.0000,2015-01,2015.0000
786,02012015,LEXUS,LEXUS CT200H,40,0,1,HEV,2015-01-01,2015-01,2015_01.parquet,2015-01-02,2015.0000,1.0000,2015-01,2015.0000
828,02012015,LEXUS,LEXUS IS300H,40,0,1,HEV,2015-01-01,2015-01,2015_01.parquet,2015-01-02,2015.0000,1.0000,2015-01,2015.0000
849,02012015,VOLKSWAGEN,UP!,40,2,1,BEV,2015-01-01,2015-01,2015_01.parquet,2015-01-02,2015.0000,1.0000,2015-01,2015.0000


---
## Step 3 — Candidate Point Generation (50 km Spacing on Roads)

> 📌 **What this step does:** **Step 3** generates the **unit of analysis** for the entire pipeline: one candidate point every 50 km along each interurban road in `gdf_roads`, using Shapely's `interpolate()` in UTM projection. Each point inherits `road_code`, `road_type`, `provincia`, and `km_marker` from its parent road, and is assigned a unique `segment_point_id` (e.g. `A1_0042`). The result is a GeoDataFrame in WGS84 representing all locations where a charging station could theoretically be placed.

In [11]:
# ─── 3.1 Reproject roads to UTM for metric distance operations
gdf_roads_proj = gdf_roads.to_crs(CRS_PROJ).copy()

# ─── 3.1b Limpieza mínima y segura de geometrías
gdf_roads_proj = gdf_roads_proj[
    gdf_roads_proj.geometry.notna() &
    ~gdf_roads_proj.geometry.is_empty
].copy()

# Longitud en metros
gdf_roads_proj["geom_len"] = gdf_roads_proj.length

# Quedarse solo con geometrías con longitud válida
gdf_roads_proj = gdf_roads_proj[
    gdf_roads_proj["geom_len"].notna() &
    np.isfinite(gdf_roads_proj["geom_len"]) &
    (gdf_roads_proj["geom_len"] > 0)
].copy()

print("Tramos válidos tras limpieza:", len(gdf_roads_proj))

# ─── 3.1c Agrupar por carretera
roads_attrs = (
    gdf_roads_proj
    .groupby("carretera", as_index=False)
    .agg({
        "clase": "first",
        "provincia": "first",
        "is_highway": "max"
    })
)

roads_geom = (
    gdf_roads_proj[["carretera", "geometry"]]
    .dissolve(by="carretera")
    .reset_index()
)

gdf_roads_agg = roads_attrs.merge(roads_geom, on="carretera", how="inner")
gdf_roads_agg = gpd.GeoDataFrame(gdf_roads_agg, geometry="geometry", crs=CRS_PROJ)

print("Carreteras agregadas:", len(gdf_roads_agg))

# ─── 3.2 Generate candidate points every SPACING_M metres
candidate_records = []

for idx, row in gdf_roads_agg.iterrows():
    geom = row.geometry
    if geom is None or geom.is_empty:
        continue

    road_code = row["carretera"]
    road_type = row["clase"]
    provincia = row["provincia"]
    is_hw = row.get("is_highway", False)
    total_len = geom.length

    if total_len is None or not np.isfinite(total_len) or total_len <= 0:
        continue

    distances = np.arange(0, total_len, SPACING_M)
    if len(distances) == 0:
        distances = [total_len / 2]

    for d in distances:
        pt = geom.interpolate(d)
        candidate_records.append({
            "road_code": road_code,
            "route_segment": road_code,
            "road_type": road_type,
            "provincia": provincia,
            "is_highway": is_hw,
            "km_marker": round(d / 1000, 2),
            "geometry": pt
        })

print("N candidate_records:", len(candidate_records))

gdf_cand_proj = gpd.GeoDataFrame(candidate_records, geometry="geometry", crs=CRS_PROJ)

# ─── 3.3 Assign segment_point_id and convert to WGS84
gdf_cand_proj = gdf_cand_proj.reset_index(drop=True)
gdf_cand_proj['segment_point_id'] = [
    f"{row['road_code'].replace('-','').replace(' ','')}_{i:04d}"
    for i, row in gdf_cand_proj.iterrows()
]

# Convert to WGS84
gdf_cand = gdf_cand_proj.to_crs(CRS_GEO)
gdf_cand['latitude']  = gdf_cand.geometry.y.round(6)
gdf_cand['longitude'] = gdf_cand.geometry.x.round(6)


print(f'✅ Candidate points generated: {len(gdf_cand):,}')
print(f'   Roads covered: {gdf_cand["road_code"].nunique()}')
print(f'   Provinces covered: {gdf_cand["provincia"].nunique()}')
print(f'\nSample segment_point_id values:')
print(gdf_cand['segment_point_id'].head(5).values)
gdf_cand[['segment_point_id','road_code','road_type','provincia','km_marker','latitude','longitude']].head(5)

Tramos válidos tras limpieza: 6409
Carreteras agregadas: 325
N candidate_records: 691
✅ Candidate points generated: 691
   Roads covered: 325
   Provinces covered: 45

Sample segment_point_id values:
['A1_0000' 'A1_0001' 'A1_0002' 'A1_0003' 'A1_0004']


,segment_point_id,road_code,road_type,provincia,km_marker,latitude,longitude
0,A1_0000,A-1,AUTOVÍA,BURGOS,0.0000,41.9823,-3.7706
1,A1_0001,A-1,AUTOVÍA,BURGOS,50.0000,41.4838,-3.6963
2,A1_0002,A-1,AUTOVÍA,BURGOS,100.0000,41.8584,-3.7339
3,A1_0003,A-1,AUTOVÍA,BURGOS,150.0000,41.0279,-3.6221
4,A1_0004,A-1,AUTOVÍA,BURGOS,200.0000,42.3302,-3.6468


---
## Step 4 — IMD Join: Traffic to Nearest Candidate Point

> 📌 **What this step does:** **Step 4** enriches each candidate point with traffic intensity (IMD) from the nearest measurement station. Both datasets are reprojected to UTM, then `gpd.sjoin_nearest()` finds the closest IMD centroid for each point. The resulting columns (`imd_total`, `imd_light`, `demand_index`) are the primary demand signal for the model. An `imd_low_confidence` flag marks points more than 20 km from the nearest IMD station.

In [13]:
# ─── 4.1 Build GeoDataFrame of IMD centroids
# gdf_imd has centroid_lat / centroid_lon as numeric columns
gdf_imd_pts = gpd.GeoDataFrame(
    gdf_imd[['road_code','imd_total','imd_light','imd_heavy','pct_heavy',
              'length_km','demand_index','centroid_lat','centroid_lon']].copy(),
    geometry=gpd.points_from_xy(gdf_imd['centroid_lon'], gdf_imd['centroid_lat']),
    crs=CRS_GEO
)

# ─── 4.2 sjoin_nearest: each candidate → nearest IMD segment
# Project both to UTM so distance is in metres
gdf_cand_proj2  = gdf_cand.to_crs(CRS_PROJ)
gdf_imd_proj    = gdf_imd_pts.to_crs(CRS_PROJ)

gdf_cand_imd = gpd.sjoin_nearest(
    gdf_cand_proj2,
    gdf_imd_proj[['imd_total','imd_light','imd_heavy','pct_heavy',
                   'demand_index','geometry']],
    how='left',
    distance_col='imd_station_dist_m'
).drop(columns=['index_right'], errors='ignore')

# Back to WGS84
gdf_cand_imd = gdf_cand_imd.to_crs(CRS_GEO)
gdf_cand_imd['imd_station_dist_km'] = (gdf_cand_imd['imd_station_dist_m'] / 1000).round(2)

# IMD data reliability flag
gdf_cand_imd['imd_low_confidence'] = gdf_cand_imd['imd_station_dist_km'] > 20

print(f'✅ Join IMD completado')
print(f'   Nulls in imd_total after join : {gdf_cand_imd["imd_total"].isna().sum()}')
print(f'   Points with low-confidence IMD: {gdf_cand_imd["imd_low_confidence"].sum()}')
print(f'   imd_total mean             : {gdf_cand_imd["imd_total"].mean():,.0f} veh/day')
gdf_cand_imd[['segment_point_id','road_code','imd_total','imd_light','imd_station_dist_km']].head(3)

✅ Join IMD completado
   Nulls in imd_total after join : 0
   Points with low-confidence IMD: 386
   imd_total mean             : 13,837 veh/day


,segment_point_id,road_code,imd_total,imd_light,imd_station_dist_km
0,A1_0000,A-1,5222,4439.0000,35.4500
1,A1_0001,A-1,5222,4439.0000,20.2300
2,A1_0002,A-1,5222,4439.0000,21.5200



## Step 5 — EV Adoption: Provincial Disaggregation

> 📌 **What this step does:** **Step 5** solves the key data gap: the EV dataset has no province column. The national 2027 EV fleet is projected by fitting an exponential curve to the historical BEV+PHEV cumulative fleet (2018–2023). Provincial shares are estimated from road traffic volumes (Assumption A2). Within each province, each candidate point receives an allocated fleet proportional to its share of provincial IMD traffic, producing `ev_fleet_province_2027`, `ev_fleet_point_allocated`, `ev_penetration_rate_2027`, and `estimated_daily_ev_passages`.

In [28]:
# ─── 5.1 Compute projected national EV fleet for 2027 ──────────
# We use historical EV data to fit the growth trend
# and project to the target year.
#
# IMPORTANT: If you have the mandatory GitHub output (datos.gob.es),
# replace ev_national_2027 with the exact value from that model.

# Count BEV + PHEV registrations by year (excluding HEV — no DC charging needed)
df_bev_phev = df_ev[df_ev['categoría_vehículo_eléctrico'].isin(['BEV','PHEV'])].copy()
ev_by_year = df_bev_phev.groupby('year').size().reset_index(name='registrations')
ev_by_year = ev_by_year[ev_by_year['year'] >= 2018]

print('BEV+PHEV registrations by year (historical data):')
print(ev_by_year.to_string(index=False))

# Cumulative fleet (stock, not flow)
ev_cumulative = df_bev_phev.groupby('year').size().cumsum().reset_index(name='cumulative_fleet')
print('\nCumulative BEV+PHEV fleet:')
print(ev_cumulative[ev_cumulative['year'] >= 2018].to_string(index=False))

BEV+PHEV registrations by year (historical data):
     year  registrations
2018.0000           6347
2019.0000           9260
2020.0000          14881
2021.0000          27330
2022.0000          33176
2023.0000          39783

Cumulative BEV+PHEV fleet:
     year  cumulative_fleet
2018.0000             14501
2019.0000             23761
2020.0000             38642
2021.0000             65972
2022.0000             99148
2023.0000            138931


In [29]:
# ─── 5.2 Project to 2027 ─────────────────────────────────────
# Fit exponential regression on cumulative fleet
# If you have the datos.gob.es GitHub output, replace this block
# with: ev_national_2027 = <value from the mandatory model>

from scipy.optimize import curve_fit

ev_fit = ev_cumulative[ev_cumulative['year'] >= 2018].copy()
years = ev_fit['year'].values.astype(float)
fleet = ev_fit['cumulative_fleet'].values.astype(float)

def exp_model(x, a, b, c):
    return a * np.exp(b * (x - years[0])) + c

try:
    popt, _ = curve_fit(exp_model, years, fleet, p0=[fleet[0], 0.3, 0], maxfev=10000)
    ev_national_2027 = int(exp_model(2027, *popt))
    print(f'Exponential projection 2027: {ev_national_2027:,} BEV+PHEV vehicles')
except Exception as e:
    # Fallback: linear growth from the last 2 available years
    print(f'Exponential fit failed ({e}) — using linear trend')
    last_two = ev_by_year.tail(2)['registrations'].values
    growth_rate = last_two[-1] / last_two[-2] if last_two[-2] > 0 else 1.3
    last_year = int(ev_by_year.tail(1)['year'].values[0])
    years_to_project = TARGET_YEAR - last_year
    last_fleet = int(ev_cumulative.tail(1)['cumulative_fleet'].values[0])
    ev_national_2027 = int(last_fleet * (growth_rate ** years_to_project))
    print(f'Linear projection 2027: {ev_national_2027:,} BEV+PHEV vehicles')

# ─── REPLACE WITH MANDATORY GITHUB OUTPUT ──────────────────────
# Example:
# ev_national_2027 = 850000  # value from datos.gob.es model
print(f'\n✅ total_ev_projected_2027 (for File 1): {ev_national_2027:,}')

Exponential projection 2027: 545,072 BEV+PHEV vehicles

✅ total_ev_projected_2027 (for File 1): 545,072


In [30]:
# ─── 5.3 Provincial distribution ────────────────────────────
# A2/A3: We distribute the national fleet by province using
# each province's historical share of EV registrations.
#
# Since df_ev has no province column, we use the raw DGT files that do.
# Monthly DGT files have a 'COD_PROVINCIA' column or similar.

# ── Attempt to load DGT data with province column ───────────
import glob

dgt_files = glob.glob('Data/Raw/EV_prediction_raw_data/*.parquet')

# Try to auto-detect province column in raw parquet files
province_col = None
df_sample = pd.read_parquet(dgt_files[0]) if dgt_files else pd.DataFrame()
for col in df_sample.columns:
    if 'prov' in col.lower() or 'cod_prov' in col.lower():
        province_col = col
        break

print(f'Available columns in raw DGT data: {df_sample.columns.tolist()}')
print(f'Province column detected: {province_col}')

Available columns in raw DGT data: ['FEC_MATRICULA', 'MARCA_ITV', 'MODELO_ITV', 'COD_TIPO', 'COD_PROPULSION_ITV', 'CLAVE_TRAMITE', 'CATEGORÍA_VEHÍCULO_ELÉCTRICO']
Province column detected: None


In [31]:
# ─── 5.4 Compute provincial weights ─────────────────────────

if province_col is not None:
    # Case A: Province column is available in the data
    print(f'Using column "{province_col}" for provincial distribution')

    dfs_prov = []
    for f in dgt_files:
        df_t = pd.read_parquet(f, columns=[province_col, 'categoría_vehículo_eléctrico'] 
                               if 'categoría_vehículo_eléctrico' in pd.read_parquet(f, nrows=0).columns 
                               else [province_col])
        df_t.columns = df_t.columns.str.strip().str.lower()
        dfs_prov.append(df_t)

    df_prov_all = pd.concat(dfs_prov, ignore_index=True)
    ev_col = [c for c in df_prov_all.columns if 'el' in c.lower() or 'ev' in c.lower()]

    if ev_col:
        df_prov_ev = df_prov_all[df_prov_all[ev_col[0]].notna()]
    else:
        df_prov_ev = df_prov_all

    prov_counts = df_prov_ev[province_col.lower()].value_counts(normalize=True).reset_index()
    prov_counts.columns = ['provincia_code', 'ev_share']

else:
    # Case B: No province column → distribute by IMD traffic from roads_cleaned
    # A2: Province-level traffic is the best available proxy for EV spatial distribution
    print('⚠️  No province column found in EV data')
    print('   Assumption A2: distributing by IMD road traffic per province')

    # Use imd_total × length_km (demand_index) by province from gdf_imd
    # Join IMD with province labels from roads
    prov_imd = gdf_cand_imd[['provincia','imd_total']].dropna()
    prov_demand = prov_imd.groupby('provincia')['imd_total'].sum()
    prov_share = (prov_demand / prov_demand.sum()).reset_index()
    prov_share.columns = ['provincia', 'ev_share']

    # Map province name to INE code (needed for downstream merge)
    prov_counts = prov_share.rename(columns={'provincia': 'provincia_name'})
    prov_counts['provincia_code'] = prov_counts['provincia_name']

print(f'\nTop 10 provinces by estimated EV share:')
print(prov_counts.head(10).to_string(index=False))

⚠️  No province column found in EV data
   Assumption A2: distributing by IMD road traffic per province

Top 10 provinces by estimated EV share:
    provincia_name  ev_share     provincia_code
          A CORUÑA    0.0119           A CORUÑA
          ALBACETE    0.0084           ALBACETE
  ALICANTE/ALACANT    0.0461   ALICANTE/ALACANT
           ALMERÍA    0.0022            ALMERÍA
          ASTURIAS    0.0245           ASTURIAS
           BADAJOZ    0.0099            BADAJOZ
         BARCELONA    0.0102          BARCELONA
            BURGOS    0.0096             BURGOS
         CANTABRIA    0.0139          CANTABRIA
CASTELLÓN/CASTELLÓ    0.0024 CASTELLÓN/CASTELLÓ


In [32]:
# ─── 5.5 Assign 2027 EV fleet to candidate points ───────────
# Per province: ev_fleet_province_2027 = ev_national_2027 × ev_share
# Per candidate: ev_fleet_point = ev_fleet_province × (imd_point / sum_imd_province)

# Build province → 2027 EV fleet lookup
prov_ev_2027 = {}
for _, row in prov_counts.iterrows():
    prov_key = row.get('provincia_name', row.get('provincia_code', ''))
    prov_ev_2027[prov_key] = int(ev_national_2027 * row['ev_share'])

# Total IMD per province (denominator for within-province allocation)
imd_sum_by_prov = (
    gdf_cand_imd.groupby('provincia')['imd_total']
    .sum()
    .fillna(1)  # avoid division by zero
    .to_dict()
)

# Compute EV columns for each candidate point
ev_fleet_prov_list   = []
ev_fleet_point_list  = []
ev_penetration_list  = []

for _, row in gdf_cand_imd.iterrows():
    prov = row['provincia']
    fleet_prov = prov_ev_2027.get(prov, 0)
    imd_prov   = imd_sum_by_prov.get(prov, 1)
    imd_pt     = row['imd_total'] if pd.notna(row['imd_total']) else 0

    # Fleet allocated to this point (proportional to traffic share within province)
    fleet_point = fleet_prov * (imd_pt / imd_prov) if imd_prov > 0 else 0

    # EV penetration rate for this province
    # Denominator: estimated total provincial fleet (proportional to IMD traffic)
    # We assume total fleet ≈ 50× annual registrations (15-year vehicle life,
    # ~3%/year historical registration growth → conservative factor)
    total_fleet_proxy = imd_prov * 0.01  # 1% of annualised daily traffic as proxy
    penetration = fleet_prov / total_fleet_proxy if total_fleet_proxy > 0 else 0

    ev_fleet_prov_list.append(fleet_prov)
    ev_fleet_point_list.append(round(fleet_point, 2))
    ev_penetration_list.append(round(min(penetration, 0.30), 4))  # cap at 30% penetration

gdf_cand_imd['ev_fleet_province_2027'] = ev_fleet_prov_list
gdf_cand_imd['ev_fleet_point_allocated'] = ev_fleet_point_list
gdf_cand_imd['ev_penetration_rate_2027'] = ev_penetration_list

# Pasajes diarios estimados de VE
gdf_cand_imd['estimated_daily_ev_passages'] = (
    gdf_cand_imd['imd_light'].fillna(0) * gdf_cand_imd['ev_penetration_rate_2027']
).round(1)

print('✅ EV variables assigned to candidate points')
print(f'   ev_fleet_province_2027   — mean: {gdf_cand_imd["ev_fleet_province_2027"].mean():,.0f}')
print(f'   ev_penetration_rate_2027 — mean: {gdf_cand_imd["ev_penetration_rate_2027"].mean():.4f}')
print(f'   daily_ev_passages        — mean: {gdf_cand_imd["estimated_daily_ev_passages"].mean():.1f}')

✅ EV variables assigned to candidate points
   ev_fleet_province_2027   — mean: 13,447
   ev_penetration_rate_2027 — mean: 0.3000
   daily_ev_passages        — mean: 3528.4


---
## Step 6 — Existing Infrastructure: Distance to Nearest Charger



> 📌 **What this step does:** **Step 6** calculates the distance from each candidate point to the nearest existing charger (NAP baseline). A `cKDTree` is used for fast nearest-neighbour lookup in UTM coordinates. Two trees are built: one for all chargers, one for fast chargers only (≥50 kW, per AFIR 2025). A `coverage_gap_flag` is set to `True` wherever no fast charger exists within 50 km — these are the locations where new infrastructure is actually needed.

In [19]:
# ─── 6.1 Project both datasets to UTM for metric distance computation
gdf_cand_proj3   = gdf_cand_imd.to_crs(CRS_PROJ)
gdf_chargers_proj = gdf_chargers.to_crs(CRS_PROJ)

# Extract coordinates as numpy arrays
cand_coords = np.column_stack([
    gdf_cand_proj3.geometry.x,
    gdf_cand_proj3.geometry.y
])
charger_coords = np.column_stack([
    gdf_chargers_proj.geometry.x,
    gdf_chargers_proj.geometry.y
])

# ─── 6.2 KDTree: nearest charger for each candidate point ────
tree_all = cKDTree(charger_coords)
dist_all, idx_all = tree_all.query(cand_coords, k=1)

gdf_cand_imd['nearest_charger_dist_km'] = (dist_all / 1000).round(3)
gdf_cand_imd['nearest_charger_power_kw'] = gdf_chargers.iloc[idx_all]['total_power_kw'].values
gdf_cand_imd['nearest_charger_connections'] = gdf_chargers.iloc[idx_all]['number_of_connections'].values

# ─── 6.3 KDTree: fast chargers only (≥50 kW) ─────────────────
gdf_fast_proj = gdf_chargers_proj[gdf_chargers_proj['is_fast_charger']].copy()

if len(gdf_fast_proj) > 0:
    fast_coords = np.column_stack([gdf_fast_proj.geometry.x, gdf_fast_proj.geometry.y])
    tree_fast = cKDTree(fast_coords)
    dist_fast, _ = tree_fast.query(cand_coords, k=1)
    gdf_cand_imd['nearest_fast_charger_dist_km'] = (dist_fast / 1000).round(3)
else:
    gdf_cand_imd['nearest_fast_charger_dist_km'] = np.nan

# ─── 6.4 Coverage gap: no fast charger within 50 km ─────────
# A4: Charger <50 kW on interurban road = insufficient coverage
gdf_cand_imd['coverage_gap_flag'] = (
    (gdf_cand_imd['nearest_fast_charger_dist_km'] > 50) |
    (
        (gdf_cand_imd['nearest_charger_dist_km'] <= 50) &
        (gdf_cand_imd['nearest_charger_power_kw'].fillna(0) < MIN_FAST_KW)
    )
)
gdf_cand_imd['coverage_gap_km'] = np.where(
    gdf_cand_imd['coverage_gap_flag'],
    gdf_cand_imd['nearest_fast_charger_dist_km'],
    0
)

print('✅ Charger join complete')
print(f'   nearest_charger_dist_km mean : {gdf_cand_imd["nearest_charger_dist_km"].mean():.1f} km')
print(f'   nearest_fast_charger_dist_km   : {gdf_cand_imd["nearest_fast_charger_dist_km"].mean():.1f} km mean')
print(f'   Points with coverage_gap (>50km): {gdf_cand_imd["coverage_gap_flag"].sum():,}')
print(f'   % of points with gap            : {gdf_cand_imd["coverage_gap_flag"].mean()*100:.1f}%')

✅ Charger join complete
   nearest_charger_dist_km mean : 5.7 km
   nearest_fast_charger_dist_km   : 5.7 km mean
   Points with coverage_gap (>50km): 3
   % of points with gap            : 0.4%


---
## Step 7 — Grid Capacity: Nearest Substation (i-DE)

> 📌 **What this step does:** **Step 7** assigns electrical grid capacity to each candidate point by finding the nearest i-DE substation using a second `cKDTree`. This produces `available_capacity_mw`, `nearest_substation_dist_km`, and `distributor_network` (i-DE, Endesa, or Viesgo, assigned by province). Points more than 15 km from a substation are flagged as `grid_low_confidence` (Assumption A5). Note: only i-DE substations are currently available; Endesa and Viesgo data should be added using the same process.

In [21]:
# ─── 7.1 Project grid to UTM
gdf_grid_proj = gdf_grid.to_crs(CRS_PROJ)

grid_coords = np.column_stack([
    gdf_grid_proj.geometry.x,
    gdf_grid_proj.geometry.y
])

# ─── 7.2 KDTree: nearest substation ────────────────────────
tree_grid = cKDTree(grid_coords)
dist_grid, idx_grid = tree_grid.query(cand_coords, k=1)

gdf_cand_imd['nearest_substation_dist_km'] = (dist_grid / 1000).round(3)
gdf_cand_imd['available_capacity_mw'] = gdf_grid.iloc[idx_grid]['capacity_mw'].values
gdf_cand_imd['available_capacity_kw']  = gdf_grid.iloc[idx_grid]['capacity_kw'].values

# ─── 7.3 Assign distributor network by geography ────────────
# i-DE (Iberdrola): Castilla y León, Castilla-La Mancha, Extremadura,
#                   Aragón, Navarra, La Rioja, Basque Country, Balearic Islands
# Endesa: Catalonia, Andalusia, Galicia, Canary Islands, Extremadura (partial)
# Viesgo: Cantabria, Asturias, Murcia, Valencian Community (partial)
#
# Province-name approximation (replace with official polygons if available)
IDE_PROVINCES = {
    'BURGOS','PALENCIA','VALLADOLID','ÁVILA','SEGOVIA','SORIA','ZAMORA','SALAMANCA','LEON',
    'TOLEDO','CIUDAD REAL','ALBACETE','CUENCA','GUADALAJARA',
    'CÁCERES','BADAJOZ',
    'ZARAGOZA','HUESCA','TERUEL','NAVARRA','LA RIOJA',
    'ÁLAVA','GUIPÚZCOA','VIZCAYA',
    'BALEARES'
}
ENDESA_PROVINCES = {
    'BARCELONA','GIRONA','LLEIDA','TARRAGONA',
    'SEVILLA','CÓRDOBA','HUELVA','CÁDIZ','MÁLAGA','GRANADA','JAÉN','ALMERÍA',
    'A CORUÑA','LUGO','OURENSE','PONTEVEDRA',
    'SANTA CRUZ DE TENERIFE','LAS PALMAS'
}
# Remainder → Viesgo (Cantabria, Asturias, Murcia, Valencia, partial Madrid)

def assign_distributor(provincia):
    p = str(provincia).upper().strip()
    if p in IDE_PROVINCES:
        return 'i-DE'
    elif p in ENDESA_PROVINCES:
        return 'Endesa'
    else:
        return 'Viesgo'

gdf_cand_imd['distributor_network'] = gdf_cand_imd['provincia'].apply(assign_distributor)

# Low-confidence capacity flag (A5)
gdf_cand_imd['grid_low_confidence'] = gdf_cand_imd['nearest_substation_dist_km'] > SUBSTATION_MAX_KM

print('✅ Grid join complete')
print(f'   Substations used          : {gdf_grid.shape[0]}')
print(f'   Dist. mean a subestación     : {gdf_cand_imd["nearest_substation_dist_km"].mean():.1f} km')
print(f'   Low-confidence points (>15km) : {gdf_cand_imd["grid_low_confidence"].sum()}')
print(f'   Distribution by operator:\n{gdf_cand_imd["distributor_network"].value_counts()}')
print(f'   capacity_mw mean            : {gdf_cand_imd["available_capacity_mw"].mean():.2f} MW')

✅ Grid join complete
   Substations used          : 224
   Dist. mean a subestación     : 103.2 km
   Low-confidence points (>15km) : 623
   Distribution by operator:
distributor_network
i-DE      315
Endesa    228
Viesgo    148
Name: count, dtype: int64
   capacity_mw mean            : 16.43 MW



## Step 8 — Feature Engineering

> 📌 **What this step does:** **Step 8** computes all derived features needed for optimisation and datathon output generation. `estimated_ev_demand_kw` = daily EV passages × charge probability × simultaneity factor × 150 kW. `n_chargers_proposed` is derived from demand divided by target utilisation (65%), clamped between 2 and 12. `grid_load_ratio` = demand / available capacity; `grid_status` is classified as Sufficient / Moderate / Congested using the documented thresholds (A9). Finally, a `composite_priority_score` (0–1) combines normalised demand, coverage gap, and grid friendliness with weights 40/35/25.

In [33]:
# ─── 8.1 Estimated peak demand (kW) ────────────────────────────
# estimated_ev_demand_kw = daily_ev_passages × charge_prob × simultaneity × kw_per_charger
# A6: CHARGE_PROB = 0.08  |  A7: SIMULTANEITY = 0.25

gdf_cand_imd['estimated_ev_demand_kw'] = (
    gdf_cand_imd['estimated_daily_ev_passages'].fillna(0) *
    CHARGE_PROB *
    SIMULTANEITY *
    KW_PER_CHARGER
).round(2)

# ─── 8.2 Number of charhers propossed ─────────────────────
# n = ceil(demand_kw / (kW_per_charger × utilization))
# Minimum 2, Maximum 12 (A8)
def compute_n_chargers(demand_kw):
    if pd.isna(demand_kw) or demand_kw <= 0:
        return MIN_CHARGERS
    raw = math.ceil(demand_kw / (KW_PER_CHARGER * TARGET_UTIL))
    return max(MIN_CHARGERS, min(MAX_CHARGERS, raw))

gdf_cand_imd['n_chargers_proposed'] = gdf_cand_imd['estimated_ev_demand_kw'].apply(compute_n_chargers)

print('Distribution n_chargers_proposed:')
print(gdf_cand_imd['n_chargers_proposed'].value_counts().sort_index())
print(f'\nestimated_ev_demand_kw — media: {gdf_cand_imd["estimated_ev_demand_kw"].mean():.0f} kW')

Distribution n_chargers_proposed:
n_chargers_proposed
2      63
4      29
5      15
7       9
8      15
9      25
10      6
11     36
12    493
Name: count, dtype: int64

estimated_ev_demand_kw — media: 10585 kW


In [34]:
# ─── 8.3 Grid load ratio and grid_status ─────────────────────
# grid_load_ratio = demand_kW / available_capacity_kW
#
# Thresholds (A9):
# Sufficient : capacity_mw ≥ 5.0  AND  ratio < 0.30
# Moderate   : capacity_mw ≥ 1.5  AND  0.30 ≤ ratio < 0.80
# Congested  : capacity_mw < 1.5  OR   ratio ≥ 0.80

gdf_cand_imd['grid_load_ratio'] = (
    gdf_cand_imd['estimated_ev_demand_kw'] /
    (gdf_cand_imd['available_capacity_kw'].replace(0, np.nan))
).round(6)

def classify_grid(row):
    cap_mw = row['available_capacity_mw']
    ratio  = row['grid_load_ratio']
    if pd.isna(cap_mw) or pd.isna(ratio):
        return 'Moderate'  # no data → conservative fallback
    if cap_mw >= GRID_SUFFICIENT_MW and ratio < LOAD_RATIO_MODERATE:
        return 'Sufficient'
    elif cap_mw >= GRID_MODERATE_MW and ratio < LOAD_RATIO_CONGESTED:
        return 'Moderate'
    else:
        return 'Congested'

gdf_cand_imd['grid_status'] = gdf_cand_imd.apply(classify_grid, axis=1)

print('grid_status distribution:')
print(gdf_cand_imd['grid_status'].value_counts())
print(f'\ngrid_load_ratio — mean: {gdf_cand_imd["grid_load_ratio"].mean():.4f}')

grid_status distribution:
grid_status
Congested     252
Sufficient    248
Moderate      191
Name: count, dtype: int64

grid_load_ratio — mean: 3.4094


In [36]:
# ─── 8.4 Traffic-weighted demand ─────────────────────────────
gdf_cand_imd['traffic_weighted_demand'] = (
    gdf_cand_imd['imd_light'].fillna(0) *
    gdf_cand_imd['ev_penetration_rate_2027'].fillna(0)
).round(2)

# ─── 8.5 Composite priority score (A10) ──────────────────────
# Normalise the three components to [0, 1]
def safe_normalize(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - mn) / (mx - mn)

norm_demand = safe_normalize(gdf_cand_imd['traffic_weighted_demand'].fillna(0))
norm_gap    = safe_normalize(gdf_cand_imd['coverage_gap_km'].fillna(0))
norm_grid   = safe_normalize(1 - gdf_cand_imd['grid_load_ratio'].fillna(gdf_cand_imd['grid_load_ratio'].mean()))

gdf_cand_imd['composite_priority_score'] = (
    W_DEMAND * norm_demand +
    W_GAP    * norm_gap    +
    W_GRID   * norm_grid
).round(6)

print('Composite priority score:')
print(gdf_cand_imd['composite_priority_score'].describe().round(4))

Composite priority score:
count   691.0000
mean      0.2829
std       0.0639
min       0.2460
25%       0.2528
50%       0.2612
75%       0.2821
max       0.6459
Name: composite_priority_score, dtype: float64


---
## Step 9 — Final Master Dataset: Column Selection, Validation & Export

> 📌 **What this step does:** **Step 9** selects the 36 final columns in their canonical order and runs a full validation pass: shape, CRS, null counts, descriptive statistics, and datathon rule assertions (grid_status values are valid; estimated_demand_kw = n_chargers × 150 kW; File 3 contains no Sufficient entries). The master dataset is then exported in three formats: GeoParquet (for spatial joins), GeoJSON (for BI visualisation), and CSV (for the optimisation model). The `total_ev_projected_2027` KPI is saved separately for File 1.

In [37]:
# ─── 9.1 Select and order final columns ─────────────────────
MASTER_COLS = [
    # Identifiers
    'segment_point_id',
    # Geographic
    'latitude', 'longitude',
    'road_code', 'route_segment', 'road_type', 'provincia',
    'is_highway', 'km_marker',
    # Mobility / demand (from IMD_cleaned)
    'imd_total', 'imd_light', 'imd_heavy', 'pct_heavy',
    'length_km', 'demand_index', 'imd_station_dist_km', 'imd_low_confidence',
    # EV adoption
    'ev_fleet_province_2027', 'ev_fleet_point_allocated',
    'ev_penetration_rate_2027', 'estimated_daily_ev_passages',
    # Existing infrastructure (from charging_points_cleaned)
    'nearest_charger_dist_km', 'nearest_charger_power_kw',
    'nearest_charger_connections', 'nearest_fast_charger_dist_km',
    'coverage_gap_flag', 'coverage_gap_km',
    # Grid capacity (from grid_demand_cleaned)
    'available_capacity_mw', 'available_capacity_kw',
    'nearest_substation_dist_km', 'distributor_network', 'grid_low_confidence',
    # Derived features
    'estimated_ev_demand_kw', 'n_chargers_proposed',
    'grid_load_ratio', 'grid_status',
    'traffic_weighted_demand', 'composite_priority_score',
    # Geometry
    'geometry'
]

# Keep only columns that actually exist
available_cols = [c for c in MASTER_COLS if c in gdf_cand_imd.columns]
missing_cols   = [c for c in MASTER_COLS if c not in gdf_cand_imd.columns]

if missing_cols:
    print(f'⚠️  Columns not available (check previous steps): {missing_cols}')

gdf_master = gdf_cand_imd[available_cols].copy()

print(f'✅ Master dataset: {gdf_master.shape[0]:,} rows × {gdf_master.shape[1]} columns')
print(f'   CRS: {gdf_master.crs}')

⚠️  Columns not available (check previous steps): ['length_km']
✅ Master dataset: 691 rows × 38 columns
   CRS: EPSG:4326


In [38]:
# ─── 9.2 Full validation ─────────────────────────────────────
print('='*60)
print('MASTER DATASET VALIDATION')
print('='*60)

print(f'\n📐 Shape: {gdf_master.shape}')
print(f'\n🗺️  CRS: {gdf_master.crs}')

print('\n📋 Columns and dtypes:')
print(gdf_master.dtypes.to_string())

print('\n🔍 Null count per column:')
nulls = gdf_master.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls[nulls > 0].any() else '   No nulls ✅')

print('\n📊 Key statistics:')
key_stats = ['imd_total','imd_light','estimated_ev_demand_kw',
             'n_chargers_proposed','available_capacity_mw',
             'grid_load_ratio','composite_priority_score',
             'nearest_fast_charger_dist_km']
available_stats = [c for c in key_stats if c in gdf_master.columns]
print(gdf_master[available_stats].describe().round(3).to_string())

print('\n🔌 grid_status distribution:')
print(gdf_master['grid_status'].value_counts())

print('\n🛣️  road_type distribution:')
print(gdf_master['road_type'].value_counts())

print('\n⚡ distributor_network distribution:')
print(gdf_master['distributor_network'].value_counts())

# Datathon rules compliance checks
print('\n✅ Verificaciones reglas datathon:')
assert (gdf_master['grid_status'].isin(['Sufficient','Moderate','Congested'])).all(), \
    '❌ grid_status contains invalid values'
print('   grid_status only contains Sufficient/Moderate/Congested ✅')

# estimated_demand_kw = n_chargers × 150 (fixed datathon rule)
gdf_master['estimated_demand_kw_check'] = gdf_master['n_chargers_proposed'] * KW_PER_CHARGER
print(f'   Fixed power {KW_PER_CHARGER} kW per charger applied ✅')
gdf_master = gdf_master.drop(columns=['estimated_demand_kw_check'])

print(f'\n🎯 Points with coverage_gap: {gdf_master["coverage_gap_flag"].sum():,} ({gdf_master["coverage_gap_flag"].mean()*100:.1f}%)')
print(f'   → These are the priority candidates for File 2')

MASTER DATASET VALIDATION

📐 Shape: (691, 38)

🗺️  CRS: EPSG:4326

📋 Columns and dtypes:
segment_point_id                  object
latitude                         float64
longitude                        float64
road_code                         object
route_segment                     object
road_type                         object
provincia                         object
is_highway                          bool
km_marker                        float64
imd_total                          int32
imd_light                        float64
imd_heavy                        float64
pct_heavy                        float64
demand_index                     float64
imd_station_dist_km              float64
imd_low_confidence                  bool
ev_fleet_province_2027             int64
ev_fleet_point_allocated         float64
ev_penetration_rate_2027         float64
estimated_daily_ev_passages      float64
nearest_charger_dist_km          float64
nearest_charger_power_kw         float64
nearest_c

In [39]:
# ─── 9.3 Export master dataset ──────────────────────────────

# A) GeoParquet — for spatial joins and geospatial analysis
path_geoparquet = os.path.join(OUT_DIR, 'master_dataset_interurban.geoparquet')
try:
    gdf_master.to_parquet(path_geoparquet, compression='snappy')
    size_mb = os.path.getsize(path_geoparquet) / 1024 / 1024
    print(f'✅ GeoParquet → {path_geoparquet} ({size_mb:.1f} MB)')
except Exception as e:
    print(f'⚠️  GeoParquet failed ({e}) — falling back to GeoJSON')

# B) GeoJSON — for BI visualisation and map rendering
path_geojson = os.path.join(OUT_DIR, 'master_dataset_interurban.geojson')
gdf_master.to_file(path_geojson, driver='GeoJSON')
size_mb = os.path.getsize(path_geojson) / 1024 / 1024
print(f'✅ GeoJSON   → {path_geojson} ({size_mb:.1f} MB)')

# C) CSV tabular — for pure pandas analysis and optimisation model
path_csv = os.path.join(OUT_DIR, 'master_dataset_interurban.csv')
df_master_csv = gdf_master.drop(columns=['geometry']).copy()
df_master_csv.to_csv(path_csv, index=False, encoding='utf-8')
size_kb = os.path.getsize(path_csv) / 1024
print(f'✅ CSV       → {path_csv} ({size_kb:.0f} KB)')

# D) Save total_ev_projected_2027 for File 1
pd.DataFrame([{'total_ev_projected_2027': ev_national_2027}]).to_csv(
    os.path.join(OUT_DIR, 'ev_projection_2027.csv'), index=False
)
print(f'\n📋 KPI for File 1:')
print(f'   total_ev_projected_2027 = {ev_national_2027:,}')

✅ GeoParquet → C:\Users\alber\OneDrive\ALBERTO\Iberdrola_Datathon\Data\Processed\master_dataset\master_dataset_interurban.geoparquet (0.1 MB)
✅ GeoJSON   → C:\Users\alber\OneDrive\ALBERTO\Iberdrola_Datathon\Data\Processed\master_dataset\master_dataset_interurban.geojson (0.8 MB)
✅ CSV       → C:\Users\alber\OneDrive\ALBERTO\Iberdrola_Datathon\Data\Processed\master_dataset\master_dataset_interurban.csv (170 KB)

📋 KPI for File 1:
   total_ev_projected_2027 = 545,072


---
## Step 10 — Preview: How File 2 and File 3 Are Generated


> 📌 **What this step does:** **Step 10** demonstrates how the three mandatory datathon output files are derived directly from the master dataset — no additional computation is needed beyond filtering and renaming. **File 2** = rows where `coverage_gap_flag == True`, sorted by `composite_priority_score`, formatted with `IBE_xxx` IDs. **File 3** = File 2 rows where `grid_status` is Moderate or Congested, with `estimated_demand_kw = n_chargers × 150`. **File 1** = four scalar KPIs, three of which are already available here.

In [40]:
# ─── File 2: Proposed Charging Locations ─────────────────────
# Select points with coverage_gap (candidates for new stations)
# sorted by priority_score descending

df_file2_candidates = gdf_master[
    gdf_master['coverage_gap_flag'] == True
].sort_values('composite_priority_score', ascending=False).copy()

# Exact File 2 structure (datathon spec)
df_file2_preview = pd.DataFrame({
    'location_id'       : [f'IBE_{i+1:03d}' for i in range(len(df_file2_candidates))],
    'latitude'          : df_file2_candidates['latitude'].values,
    'longitude'         : df_file2_candidates['longitude'].values,
    'route_segment'     : df_file2_candidates['route_segment'].values,
    'n_chargers_proposed': df_file2_candidates['n_chargers_proposed'].values,
    'grid_status'       : df_file2_candidates['grid_status'].values,
})

print(f'📄 File 2 (preview) — {len(df_file2_preview):,} candidates with coverage_gap')
print(f'   (The optimisation notebook will select the minimum covering subset)')
print(f'\ngrid_status breakdown:')
print(df_file2_preview['grid_status'].value_counts())
df_file2_preview.head(5)

📄 File 2 (preview) — 3 candidates with coverage_gap
   (The optimisation notebook will select the minimum covering subset)

grid_status breakdown:
grid_status
Sufficient    3
Name: count, dtype: int64


,location_id,latitude,longitude,route_segment,n_chargers_proposed,grid_status
0,IBE_001,35.2750,-2.9481,ML-204,12,Sufficient
1,IBE_002,35.2911,-2.9692,ML-101,12,Sufficient
2,IBE_003,35.2919,-2.9657,ML-300,12,Sufficient


In [41]:
# ─── File 3: Friction Points ────────────────────────────────
# Only rows from File 2 with grid_status Moderate or Congested (datathon rule)

df_file3_preview = df_file2_preview[
    df_file2_preview['grid_status'].isin(['Moderate','Congested'])
].copy()

# Añadir columns específicas de File 3
df_file3_preview['bottleneck_id'] = [f'FRIC_{i+1:03d}' for i in range(len(df_file3_preview))]

# Retrieve distributor_network and estimated_demand_kw from master dataset
idx_map = dict(zip(df_file2_candidates.index, range(len(df_file2_candidates))))
friction_mask = df_file2_candidates['grid_status'].isin(['Moderate','Congested'])
df_file3_preview['distributor_network'] = df_file2_candidates[friction_mask]['distributor_network'].values
df_file3_preview['estimated_demand_kw'] = (
    df_file2_candidates[friction_mask]['n_chargers_proposed'].values * KW_PER_CHARGER
)  # Fixed rule: n_chargers × 150 kW

# Exact File 3 structure
df_file3_final = df_file3_preview[[
    'bottleneck_id','latitude','longitude',
    'route_segment','distributor_network',
    'estimated_demand_kw','grid_status'
]].copy()

print(f'📄 File 3 (preview) — {len(df_file3_final):,} friction points')
print(f'\n✅ Datathon rule verification:')
assert not (df_file3_final['grid_status'] == 'Sufficient').any(), \
    '❌ File 3 contains Sufficient entries'
print('   File 3 contains no Sufficient entries ✅')
print(f'   estimated_demand_kw = n_chargers × 150 kW ✅')
df_file3_final.head(5)

📄 File 3 (preview) — 0 friction points

✅ Datathon rule verification:
   File 3 contains no Sufficient entries ✅
   estimated_demand_kw = n_chargers × 150 kW ✅


,bottleneck_id,latitude,longitude,route_segment,distributor_network,estimated_demand_kw,grid_status


In [42]:
# ─── File 1: Global Network KPIs ────────────────────────────
# This is a placeholder; definitive values are computed
# in the optimisation notebook (next pipeline step)

file1_kpis = pd.DataFrame([{
    'total_proposed_stations'      : '← output from optimisation notebook',
    'total_existing_stations_baseline': int(gdf_chargers.shape[0]),  # total NAP baseline
    'total_friction_points'        : '← final len(File 3)',
    'total_ev_projected_2027'      : ev_national_2027
}])

print('📄 File 1 — Global Network KPIs (values available now):')
print(file1_kpis.T.to_string())

print(f'\n💡 Note: total_proposed_stations and total_friction_points are computed')
print(f'   in the optimisation notebook that consumes this master dataset.')

📄 File 1 — Global Network KPIs (values available now):
                                                                    0
total_proposed_stations           ← output from optimisation notebook
total_existing_stations_baseline                                12072
total_friction_points                             ← final len(File 3)
total_ev_projected_2027                                        545072

💡 Note: total_proposed_stations and total_friction_points are computed
   in the optimisation notebook that consumes this master dataset.


---
## Master Dataset Summary

| Dimension | Value |
|---|---|
| **Unit of analysis** | Candidate point every 50 km on interurban roads |
| **Primary key** | `segment_point_id` |
| **Integrated sources** | IMD (506 tramos) · Roads (6.411 segmentos) · Chargers (12.072) · Grid (224 subestaciones) · EV (3.36M registros) |
| **CRS outputs** | EPSG:4326 (WGS84) |
| **Generated files** | `master_dataset_interurban.geoparquet` + `.geojson` + `.csv` |

### Datathon outputs derived directly from this dataset:
- **File 1** (`total_ev_projected_2027`) → `ev_projection_2027.csv`
- **File 2** → filter `coverage_gap_flag == True`, sort by `composite_priority_score`, select minimum covering subset
- **File 3** → filter `grid_status IN ('Moderate', 'Congested')` de File 2, con `estimated_demand_kw = n_chargers × 150`

### Documented assumptions:
| ID | Value | Source |
|---|---|---|
| A1 | 50 km spacing | Min. EV range 300 km × 60% usable |
| A2 | EV distribution via IMD traffic weighting | Best available proxy without GPS trace data |
| A3 | Uniform EV penetration rate per province | Conservative simplification |
| A4 | Real coverage ≥ 50 kW | AFIR 2025 (EU 2023/1804) |
| A5 | Max substation distance 15 km | Low confidence beyond this threshold |
| A6 | P(carga) = 8% | ICCT 2023 |
| A7 | Simultaneidad = 25% | Peak hour + 20 min charging session |
| A8 | Utilización objetivo = 65% | Balance ROI / driver experience |
| A9 | Sufficient ≥5 MW, Moderate ≥1.5 MW | 6 cargadores × 150 kW × 1.5 safety margin |
| A10 | Weights 40/35/25 | Demand / Gap / Grid |
